# 03 已確認上車點的目的地排序
在歷史候選上加入本次已確認上車位置與附近歷史。離線使用答案行程的實際上車座標，只對應產品中已確認上車點之後的階段，不代表開啟 App 前就知道位置。
真實競賽資料與逐筆結果未包含於公開版本；作者同意公開的彙總命中率列於文末。

快照以訂單建立時間切分，歷史皆嚴格早於 cutoff。每段答案是該期間每帳號第一趟，且快照前至少 3 趟。

| 快照 | cutoff（含於答案期間） | 答案期間結束（不含） |
| --- | --- | --- |
| train_a | 2026-02-15 | 2026-03-01 |
| train_b | 2026-03-01 | 2026-03-16 |
| validation | 2026-03-16 | 2026-04-01 |
| april | 2026-04-01 | 2026-05-01 |

同一帳號可出現在不同快照，但候選與特徵每次只由該快照之前資料建立。網格沿用 01 的固定近似 500 公尺換算與邊界限制。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
CSV_PATH = Path('../data/yoxi_cleaned.csv')
if not CSV_PATH.is_file():
    raise FileNotFoundError(f'找不到清理檔：{CSV_PATH}')
COLUMNS = ['Id', 'RiderId', 'OrderFormCreatedTime_UTC8', 'PickUpLatitude', 'PickUpLongitude',
    'DropOffLatitude', 'DropOffLongitude']
trips = pd.read_csv(CSV_PATH, usecols=COLUMNS, encoding='utf-8-sig', dtype={'Id': 'string',
    'RiderId': 'string'}, low_memory=False)
T = 'OrderFormCreatedTime_UTC8'
trips[T] = pd.to_datetime(trips[T], errors='coerce')
for c in ('PickUpLatitude', 'PickUpLongitude', 'DropOffLatitude', 'DropOffLongitude'):
    trips[c] = pd.to_numeric(trips[c], errors='coerce')
if trips[COLUMNS].isna().any().any() or trips['Id'].duplicated().any():
    raise ValueError('必要欄位有缺值或 Id 重複，請回到清理檔核對；03 保留原資料，請先核對輸入。')
CELL_M = 500
LAT_METERS_PER_DEG = 111320.0
LON_METERS_PER_DEG = 111320.0 * np.cos(np.deg2rad(24.5))

def make_cell(lat, lon, cell_m=CELL_M):
    x = np.floor(pd.to_numeric(lon, errors='coerce') * LON_METERS_PER_DEG / cell_m)
    y = np.floor(pd.to_numeric(lat, errors='coerce') * LAT_METERS_PER_DEG / cell_m)
    return 'g' + str(cell_m) + '_' + x.astype('Int64').astype('string') + '_' + y.astype('Int64').astype('string')
trips['dropoff_area'] = make_cell(trips['DropOffLatitude'], trips['DropOffLongitude'])
trips = trips.sort_values([T, 'Id'], kind='stable').reset_index(drop=True)
print(f'行程 {len(trips):,} 筆；下車網格 {trips.dropoff_area.nunique():,} 個')
WINDOWS = [('train_a', '2026-02-15', '2026-03-01'), ('train_b', '2026-03-01', '2026-03-16'), ('validation',
    '2026-03-16', '2026-04-01'), ('april', '2026-04-01', '2026-05-01')]

def historical_snapshot(all_trips, cutoff, end):
    cutoff, end = (pd.Timestamp(cutoff), pd.Timestamp(end))
    history = all_trips.loc[all_trips[T].lt(cutoff)]
    future = all_trips.loc[all_trips[T].ge(cutoff) & all_trips[T].lt(end)]
    first = future.sort_values(['RiderId', T, 'Id'], kind='stable').drop_duplicates('RiderId',
        keep='first')[['RiderId', 'PickUpLatitude', 'PickUpLongitude',
        'dropoff_area']].rename(columns={'PickUpLatitude': 'current_pickup_lat',
        'PickUpLongitude': 'current_pickup_lon', 'dropoff_area': 'actual_area'})
    counts = history.groupby('RiderId', observed=True).size().rename('history_rides')
    answers = first.join(counts, on='RiderId')
    answers = answers.loc[answers['history_rides'].ge(3)].copy()
    return (history, answers)


## 附近歷史、候選與特徵
750 公尺搜尋比較「同一帳號的歷史上車座標」與「本次已確認上車座標」，找出附近歷史行程，再依其下車網格統計。這和 01 比較預測目的地與實際下車點的容差用途不同。
候選為個人頻率 Top-10、附近上車歷史目的地 Top-3、全體熱門 Top-3 的聯集。位置特徵包含附近趟數、目的地次數與占比、最近日期、最近歷史上車距離及附近排名旗標。未知距離補 9999 公尺，未知天數補 999。

In [ ]:
RADIUS_M = 750
BASE_FEATURES = ['log_user_rides', 'active_weeks', 'user_recency_days', 'log_dest_rides', 'personal_share',
    'personal_rank', 'dest_recency_days', 'log_global_riders', 'log_global_rides', 'is_personal_top10',
    'is_global_top3']
PICKUP_FEATURES = ['log_nearby_rides', 'log_nearby_dest_rides', 'nearby_dest_share',
    'nearby_dest_recency_days', 'nearest_pickup_m', 'is_nearby_top3']

def haversine_m(lat1, lon1, lat2, lon2):
    a1, o1, a2, o2 = [np.deg2rad(np.asarray(v, dtype='float64')) for v in (lat1, lon1, lat2, lon2)]
    dlat, dlon = (a2 - a1, o2 - o1)
    h = np.sin(dlat / 2) ** 2 + np.cos(a1) * np.cos(a2) * np.sin(dlon / 2) ** 2
    return 2 * 6371000 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

def build_snapshot(history, answers, cutoff):
    if answers.empty:
        raise ValueError('此快照沒有符合條件帳號。')
    cutoff = pd.Timestamp(cutoff)
    ids = answers['RiderId']
    hist = history.loc[history['RiderId'].isin(ids), ['RiderId', T, 'PickUpLatitude', 'PickUpLongitude',
        'dropoff_area']]
    user = hist.groupby('RiderId', observed=True).agg(user_rides=('dropoff_area', 'size'), user_last=(T,
        'max'))
    user['active_weeks'] = hist.assign(week=hist[T].dt.to_period('W')).groupby('RiderId',
        observed=True)['week'].nunique()
    user['log_user_rides'] = np.log1p(user['user_rides'])
    user['user_recency_days'] = ((cutoff - user['user_last']).dt.total_seconds() / 86400).clip(lower=0)
    user = user[['user_rides', 'active_weeks', 'log_user_rides', 'user_recency_days']].reset_index()
    personal = hist.groupby(['RiderId', 'dropoff_area'], observed=True).agg(dest_rides=('dropoff_area',
        'size'), dest_last=(T, 'max')).reset_index().sort_values(['RiderId', 'dest_rides', 'dropoff_area'],
        ascending=[True, False, True], kind='stable')
    personal['personal_rank'] = personal.groupby('RiderId', observed=True).cumcount() + 1
    personal['is_personal_top10'] = personal['personal_rank'].le(10).astype('int8')
    near = hist.merge(answers[['RiderId', 'current_pickup_lat', 'current_pickup_lon']], on='RiderId',
        how='inner', validate='many_to_one')
    near['pickup_gap_m'] = haversine_m(near['PickUpLatitude'], near['PickUpLongitude'],
        near['current_pickup_lat'], near['current_pickup_lon'])
    near = near.loc[near['pickup_gap_m'].le(RADIUS_M)]
    nearby_user = near.groupby('RiderId', observed=True).size().rename('nearby_rides')
    nearby = near.groupby(['RiderId', 'dropoff_area'], observed=True).agg(nearby_dest_rides=('dropoff_area',
        'size'), nearby_dest_last=(T, 'max'), nearest_pickup_m=('pickup_gap_m',
        'min')).reset_index().sort_values(['RiderId', 'nearby_dest_rides', 'dropoff_area'], ascending=[True,
        False, True], kind='stable')
    nearby['nearby_rank'] = nearby.groupby('RiderId', observed=True).cumcount() + 1
    nearby['is_nearby_top3'] = nearby['nearby_rank'].le(3).astype('int8')
    global_stats = history.groupby('dropoff_area', observed=True).agg(global_riders=('RiderId', 'nunique'),
        global_rides=('RiderId', 'size')).reset_index().sort_values(['global_riders', 'global_rides',
        'dropoff_area'], ascending=[False, False, True], kind='stable')
    global_stats['global_rank'] = np.arange(1, len(global_stats) + 1)
    global_top = global_stats.head(3)['dropoff_area'].tolist()
    global_candidates = pd.MultiIndex.from_product([ids.tolist(), global_top], names=['RiderId',
        'dropoff_area']).to_frame(index=False)
    candidate_keys = pd.concat([personal.loc[personal['is_personal_top10'].eq(1), ['RiderId',
        'dropoff_area']], nearby.loc[nearby['is_nearby_top3'].eq(1), ['RiderId', 'dropoff_area']],
        global_candidates], ignore_index=True).drop_duplicates(['RiderId', 'dropoff_area'])
    out = candidate_keys.merge(personal, on=['RiderId', 'dropoff_area'], how='left',
        validate='one_to_one').merge(nearby, on=['RiderId', 'dropoff_area'], how='left',
        validate='one_to_one').merge(global_stats, on='dropoff_area', how='left',
        validate='many_to_one').merge(user, on='RiderId', how='left',
        validate='many_to_one').merge(nearby_user.reset_index(), on='RiderId', how='left',
        validate='many_to_one').merge(answers[['RiderId', 'actual_area']], on='RiderId', how='left',
        validate='many_to_one')
    out['dest_rides'] = out['dest_rides'].fillna(0)
    out['personal_rank'] = out['personal_rank'].fillna(999)
    out['is_personal_top10'] = out['is_personal_top10'].fillna(0)
    out['personal_share'] = out['dest_rides'] / out['user_rides']
    out['dest_recency_days'] = ((cutoff - out['dest_last']).dt.total_seconds() / 86400).clip(lower=0).fillna(999)
    out['log_dest_rides'] = np.log1p(out['dest_rides'])
    out['log_global_riders'] = np.log1p(out['global_riders'])
    out['log_global_rides'] = np.log1p(out['global_rides'])
    out['is_global_top3'] = out['global_rank'].le(3).astype('int8')
    out['nearby_rides'] = out['nearby_rides'].fillna(0)
    out['nearby_dest_rides'] = out['nearby_dest_rides'].fillna(0)
    out['log_nearby_rides'] = np.log1p(out['nearby_rides'])
    out['log_nearby_dest_rides'] = np.log1p(out['nearby_dest_rides'])
    out['nearby_dest_share'] = (out['nearby_dest_rides'] / out['nearby_rides'].replace(0, np.nan)).fillna(0)
    out['nearby_dest_recency_days'] = ((cutoff - out['nearby_dest_last']).dt.total_seconds() / 86400).clip(lower=0).fillna(999)
    out['nearest_pickup_m'] = out['nearest_pickup_m'].fillna(9999)
    out['is_nearby_top3'] = out['is_nearby_top3'].fillna(0)
    out['y'] = out['dropoff_area'].eq(out['actual_area']).astype('int8')
    if out[BASE_FEATURES + PICKUP_FEATURES].isna().any().any():
        raise AssertionError('候選特徵有缺值。')
    if out.duplicated(['RiderId', 'dropoff_area']).any() or out.groupby('RiderId')['y'].sum().gt(1).any():
        raise AssertionError('候選或正例重複。')
    for c in BASE_FEATURES + PICKUP_FEATURES:
        out[c] = out[c].astype('float32')
    return out
snapshots = {}
for name, cutoff, end in WINDOWS:
    h, a = historical_snapshot(trips, cutoff, end)
    data = build_snapshot(h, a, cutoff)
    snapshots[name] = data
    print(f"{name}: {len(a):,} 位；候選 {len(data):,}；候選涵蓋 {data.y.sum() / len(a):.2%}；有附近歷史 {data.loc[data.nearby_rides.gt(0), 'RiderId'].nunique():,} 位")
    del h, a


## 對照與分母
比較個人頻率規則、附近歷史位置規則、基礎特徵 ML、加位置特徵 ML。位置規則先依附近目的地次數，再依個人次數排序。
兩個 ML 共用含位置資訊的候選池；「基礎特徵」只表示評分欄位不含新增位置特徵，不能稱為完全不知道位置的對照。比較可觀察位置特徵的增量，但不能單獨估計整個位置候選機制的貢獻。
Top-1／Top-3 為目的地網格命中，分母包含候選未涵蓋答案的所有合資格帳號。分數未校準，不是真實出行機率。

In [ ]:
def rank_metrics(frame, sort_cols, ascending, selected=None):
    d = frame if selected is None else frame.loc[selected]
    n = frame['RiderId'].nunique()
    d = d.sort_values(['RiderId'] + sort_cols + ['dropoff_area'], ascending=[True] + ascending + [True],
        kind='stable').copy()
    d['rank'] = d.groupby('RiderId', observed=True).cumcount() + 1
    return (d.loc[d['rank'].eq(1), 'y'].sum() / n, d.loc[d['rank'].le(3)].groupby('RiderId',
        observed=True)['y'].max().sum() / n)

def evaluate(frame, scores_base=None, scores_pickup=None):
    n = frame.RiderId.nunique()
    p1, p3 = rank_metrics(frame, ['dest_rides'], [False], frame.is_personal_top10.eq(1))
    c1, c3 = rank_metrics(frame, ['nearby_dest_rides', 'dest_rides'], [False, False],
        frame.is_nearby_top3.eq(1) | frame.is_personal_top10.eq(1))
    row = {'accounts': n, 'candidate_coverage': frame.y.sum() / n, 'personal_top1': p1, 'personal_top3': p3,
        'pickup_rule_top1': c1, 'pickup_rule_top3': c3}
    if scores_base is not None:
        tmp = frame.assign(score=np.asarray(scores_base))
        row['base_model_top1'], row['base_model_top3'] = rank_metrics(tmp, ['score'], [False])
    if scores_pickup is not None:
        tmp = frame.assign(score=np.asarray(scores_pickup))
        row['pickup_model_top1'], row['pickup_model_top3'] = rank_metrics(tmp, ['score'], [False])
    return row
for name in ('validation', 'april'):
    print(name, evaluate(snapshots[name]))


## 固定設定與三月驗證
沿用原有固定 HistGradientBoostingClassifier 設定，以兩個歷史快照訓練。兩組模型使用相同參數，比較基礎特徵與新增位置特徵的差異；位置規則另作對照。

In [ ]:
PARAMS = dict(max_iter=140, max_leaf_nodes=31, min_samples_leaf=40, learning_rate=0.06,
    l2_regularization=1.0, early_stopping=False, random_state=42)
train = pd.concat([snapshots['train_a'], snapshots['train_b']], ignore_index=True)
valid = snapshots['validation']
if train.y.nunique() != 2:
    raise ValueError('訓練候選需要正例和負例。')
base_model = HistGradientBoostingClassifier(**PARAMS).fit(train[BASE_FEATURES], train.y)
pickup = HistGradientBoostingClassifier(**PARAMS).fit(train[BASE_FEATURES + PICKUP_FEATURES], train.y)
valid_result = evaluate(valid, base_model.predict_proba(valid[BASE_FEATURES])[:, 1],
    pickup.predict_proba(valid[BASE_FEATURES + PICKUP_FEATURES])[:, 1])
display(pd.DataFrame([valid_result], index=['validation']).T.rename(columns={'validation': '數值'}))


## 四月回測
合併訓練與三月驗證候選樣本重新 fit，四月答案不進入訓練。四月已用於先前探索，此回測不能宣稱為未接觸的最終測試。
結果只適用於有足夠歷史、期間內有下一趟且已確認上車點的帳號，不含線上點擊、叫車或轉換效益。

In [ ]:
pre_april = pd.concat([train, valid], ignore_index=True)
final_base = HistGradientBoostingClassifier(**PARAMS).fit(pre_april[BASE_FEATURES], pre_april.y)
final_pickup = HistGradientBoostingClassifier(**PARAMS).fit(pre_april[BASE_FEATURES + PICKUP_FEATURES],
    pre_april.y)
april = snapshots['april']
april_result = evaluate(april, final_base.predict_proba(april[BASE_FEATURES])[:, 1],
    final_pickup.predict_proba(april[BASE_FEATURES + PICKUP_FEATURES])[:, 1])
display(pd.DataFrame([april_result], index=['april']).T.rename(columns={'april': '數值'}))
print('同候選池：加入位置後 Top-3 差距（百分點）',
    f"{100 * (april_result['pickup_model_top3'] - april_result['base_model_top3']):+.2f}")
print('相對個人熱門規則 Top-3 差距（百分點）',
    f"{100 * (april_result['pickup_model_top3'] - april_result['personal_top3']):+.2f}")


## 已記錄的評估結果

以下整理自作者既有的本機執行回報，本次未讀取資料集、重新訓練或重算。只公開彙總命中率，不附帳號數、實際位置與逐筆預測。

| 方法 | 三月驗證 Top-1 | 三月驗證 Top-3 | 四月回測 Top-1 | 四月回測 Top-3 |
| --- | ---: | ---: | ---: | ---: |
| 個人最常去規則 | 32.14% | 48.45% | 29.76% | 45.77% |
| 上車附近歷史規則 | 40.07% | 50.55% | 37.38% | 47.91% |
| 基礎特徵 ML（共用位置候選池） | 33.04% | 48.97% | 30.51% | 46.23% |
| 加入位置特徵的 ML | 41.07% | 51.22% | 38.49% | 48.49% |

四月位置 ML 相對個人規則，Top-1 約增加 8.73 個百分點、Top-3 約增加 2.72 個百分點；相對位置規則則分別約增加 1.11、0.58 個百分點。增益的一部分來自位置資訊，不能全部歸因於 ML。相對共用候選池的基礎特徵 ML，新增位置特徵的 Top-3 增量約為 2.26 個百分點。

所有命中率均以目的地網格評估，包含候選未涵蓋答案的帳號。位置模型以已確認上車點為前提；四月已參與前期探索，需以未看過的期間再驗證是否可推廣。Demo 的捷徑仍為人工示例，未接上這些模型。
